In [ ]:
import gc
import random
import re
import shutil
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from skimage.util import img_as_float32
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from ultralytics import YOLO

In [ ]:
@dataclass
class PipelineConfig:

    base_dir: Path = field(default_factory=Path.cwd)
    dataset_name: str = "ankhe_dataset"
    image_size: Tuple[int, int] = (320, 320)

    train_ratio: float = 0.6
    val_ratio: float = 0.15
    test_ratio: float = 0.25

    convlstm_epochs: int = 40
    convlstm_batch_size: int = 1
    convlstm_learning_rate: float = 5e-2
    convlstm_sequence_length: int = 5
    convlstm_accumulation_steps: int = 2

    yolo_epochs: int = 100
    yolo_image_size: int = 320

    pixel_area_km2: float = 0.0001
    seed: int = 42

    def __post_init__(self):
        self.raw_image_dir = self.base_dir / self.dataset_name / "images"
        self.raw_mask_dir = self.base_dir / self.dataset_name / "masks"
        self.processed_dir = self.base_dir / "processed"
        self.yolo_image_dir = self.processed_dir / "images"
        self.yolo_label_dir = self.processed_dir / "labels"
        self.convlstm_mask_dir = self.processed_dir / "masks"
        self.dense_mask_dir = self.processed_dir / "dense_masks"
        self.split_dir = self.processed_dir / "splits"

    def create_directories(self):
        for d in [self.processed_dir, self.yolo_image_dir, self.yolo_label_dir,
                  self.convlstm_mask_dir, self.dense_mask_dir, self.split_dir]:
            d.mkdir(parents=True, exist_ok=True)

    def set_random_seeds(self):
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)


cfg = PipelineConfig()
cfg.create_directories()
cfg.set_random_seeds()

print(f"Dataset: {cfg.raw_image_dir}")

In [ ]:
def apply_mmse_filter(image: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    channels = cv2.split(image) if image.ndim == 3 else [image]
    filtered = []

    for ch in channels:
        ch_float = img_as_float32(ch)
        squared_blur = cv2.GaussianBlur(ch_float ** 2, (kernel_size, kernel_size), 0)
        blur_squared = cv2.GaussianBlur(ch_float, (kernel_size, kernel_size), 0) ** 2
        noise_var = np.mean(squared_blur - blur_squared)

        local_mean = cv2.GaussianBlur(ch_float, (kernel_size, kernel_size), 0)
        local_var = np.maximum(squared_blur - blur_squared, 1e-6)
        signal_var = np.maximum(local_var - noise_var, 0)

        wiener = signal_var / local_var
        result = local_mean + wiener * (ch_float - local_mean)
        filtered.append(np.clip(result, 0, 1))

    return cv2.merge(filtered) if len(filtered) > 1 else filtered[0]

In [ ]:
def preprocess_satellite_imagery(config: PipelineConfig) -> List[Dict[str, Path]]:
    records = []

    for img_path in sorted(config.raw_image_dir.glob("*.png")):
        mask_path = config.raw_mask_dir / img_path.name
        if not mask_path.exists():
            continue

        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        image = apply_mmse_filter(img_as_float32(image))
        image = cv2.resize(image, config.image_size, interpolation=cv2.INTER_AREA)

        out_img = config.yolo_image_dir / img_path.name
        cv2.imwrite(str(out_img), (np.clip(image, 0, 1) * 255).astype(np.uint8))

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, config.image_size, interpolation=cv2.INTER_NEAREST)
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

        out_mask = config.convlstm_mask_dir / mask_path.name
        cv2.imwrite(str(out_mask), mask)

        records.append({"img": out_img, "mask": out_mask})

    return records


records = preprocess_satellite_imagery(cfg)
print(f"Processed {len(records)} image-mask pairs")

In [ ]:
def convert_mask_to_yolo_polygon(mask_path: Path, image_size: Tuple[int, int] = (320, 320),
                                  max_points: int = 200) -> Optional[List[float]]:
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    if not contours:
        return None

    contour = max(contours, key=cv2.contourArea)
    if cv2.contourArea(contour) < 100:
        return None

    epsilon = 0.001 * cv2.arcLength(contour, True)
    simplified = cv2.approxPolyDP(contour, epsilon, True)

    if len(simplified) > max_points:
        indices = np.linspace(0, len(simplified) - 1, max_points, dtype=int)
        simplified = simplified[indices]

    if len(simplified) < 3:
        return None

    coords = []
    for pt in simplified.flatten().reshape(-1, 2):
        coords.extend([np.clip(pt[0] / image_size[0], 0, 1),
                       np.clip(pt[1] / image_size[1], 0, 1)])
    return coords


def export_yolo_labels(records: List[Dict[str, Path]], output_dir: Path) -> int:
    output_dir.mkdir(parents=True, exist_ok=True)
    count = 0

    for record in records:
        coords = convert_mask_to_yolo_polygon(record["mask"])
        if coords and len(coords) >= 6:
            label_path = output_dir / (Path(record["img"]).stem + ".txt")
            with open(label_path, "w") as f:
                f.write(f"0 {' '.join(f'{c:.6f}' for c in coords)}")
            count += 1

    print(f"Generated {count} YOLO labels")
    return count


export_yolo_labels(records, cfg.yolo_label_dir)

In [ ]:
def parse_timestamp(file_path: Path) -> Optional[pd.Timestamp]:
    match = re.search(r"(20\d{2})[-_]?([01]?\d)", file_path.stem)
    if match:
        year, month = int(match.group(1)), int(match.group(2))
        if 1 <= month <= 12:
            return pd.Timestamp(year=year, month=month, day=1)
    return None


def sort_records_by_timestamp(records: List[Dict[str, Path]]) -> List[Dict[str, Path]]:
    return sorted(records, key=lambda r: (parse_timestamp(Path(r["mask"])) or pd.Timestamp.max, r["mask"].name))

In [ ]:
def write_split_file(paths: List[Path], output: Path):
    with open(output, "w") as f:
        f.writelines(f"{p}\n" for p in paths)


def load_split_file(path: Path) -> List[Path]:
    if not path.exists():
        return []
    with open(path) as f:
        return [Path(line.strip()) for line in f if line.strip()]


def create_dataset_splits(records: List[Dict[str, Path]], config: PipelineConfig,
                          dense_masks: Optional[List[Path]] = None,
                          interpolated_masks: Optional[set] = None) -> Dict[str, List[Path]]:
    sorted_records = sort_records_by_timestamp(records)
    images = [config.yolo_image_dir / Path(r["img"]).name for r in sorted_records]

    train, temp = train_test_split(images, train_size=config.train_ratio, shuffle=False)
    val, test = train_test_split(temp, test_size=config.test_ratio / (config.val_ratio + config.test_ratio), shuffle=False)

    splits = {"train": train, "val": val, "test": test}
    write_split_file(train, config.split_dir / "train.txt")
    write_split_file(val, config.split_dir / "val.txt")
    write_split_file(test, config.split_dir / "test.txt")

    if dense_masks:
        sorted_masks = sorted(dense_masks, key=lambda p: (parse_timestamp(p) or pd.Timestamp.max, p.name))
        interpolated_set = interpolated_masks or set()

        real_masks = [m for m in sorted_masks if m not in interpolated_set]

        real_train, real_temp = train_test_split(real_masks, train_size=config.train_ratio, shuffle=False)
        real_val, real_test = train_test_split(real_temp,
                                                test_size=config.test_ratio / (config.val_ratio + config.test_ratio),
                                                shuffle=False)

        train_timestamps = {parse_timestamp(p) for p in real_train if parse_timestamp(p) is not None}
        if train_timestamps:
            train_end_ts = max(train_timestamps)
        else:
            train_end_ts = pd.Timestamp.min

        train_interp = [m for m in sorted_masks
                        if m in interpolated_set and
                        (parse_timestamp(m) is not None and parse_timestamp(m) <= train_end_ts)]

        train_m = sorted(real_train + train_interp,
                         key=lambda p: (parse_timestamp(p) or pd.Timestamp.max, p.name))

        val_m = real_val
        test_m = real_test

        splits.update({"train_masks": train_m, "val_masks": val_m, "test_masks": test_m})
        write_split_file(train_m, config.split_dir / "train_masks.txt")
        write_split_file(val_m, config.split_dir / "val_masks.txt")
        write_split_file(test_m, config.split_dir / "test_masks.txt")

        n_train_real = len(real_train)
        n_train_interp = len(train_interp)
        print(f"\n=== DATA SPLIT SUMMARY (No Leakage) ===")
        print(f"Training masks: {len(train_m)} total ({n_train_real} real + {n_train_interp} interpolated)")
        print(f"Validation masks: {len(val_m)} (ALL REAL - no interpolated)")
        print(f"Test masks: {len(test_m)} (ALL REAL - no interpolated)")
        print(f"========================================\n")

    print(f"Splits: {{{', '.join(f'{k}: {len(v)}' for k, v in splits.items())}}}")
    return splits


splits = None

In [ ]:
def augment_image_mask(image: np.ndarray, mask: np.ndarray) -> List[Tuple[np.ndarray, np.ndarray]]:
    return [
        (cv2.flip(image, 1), cv2.flip(mask, 1)),
        (cv2.flip(image, 0), cv2.flip(mask, 0)),
        (cv2.flip(image, -1), cv2.flip(mask, -1)),
        (cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE), cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)),
        (cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE), cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)),
    ]


def augment_training_data(train_records: List[Dict[str, Path]], all_records: List[Dict[str, Path]],
                          config: PipelineConfig) -> List[Dict[str, Path]]:
    augmented = []

    for record in tqdm(train_records, desc="Augmenting"):
        image = cv2.cvtColor(cv2.imread(str(record["img"])), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(record["mask"]), cv2.IMREAD_GRAYSCALE)
        stem = Path(record["img"]).stem

        for i, (aug_img, aug_mask) in enumerate(augment_image_mask(image, mask)):
            out_img = config.yolo_image_dir / f"{stem}_aug{i}.png"
            out_mask = config.convlstm_mask_dir / f"{stem}_aug{i}.png"
            cv2.imwrite(str(out_img), cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(out_mask), aug_mask)
            augmented.append({"img": out_img, "mask": out_mask, "is_augmented": True})

    print(f"Generated {len(augmented)} augmented samples")
    return all_records + augmented

In [ ]:
torch.cuda.empty_cache()
gc.collect()

loss_fn = nn.MSELoss()

def pearson_interpolation(m1: np.ndarray, m2: np.ndarray, alpha: float) -> np.ndarray:
    m1_float = m1.astype(np.float32) / 255.0 if m1.max() > 1 else m1.astype(np.float32)
    m2_float = m2.astype(np.float32) / 255.0 if m2.max() > 1 else m2.astype(np.float32)
    interpolated = (1 - alpha) * m1_float + alpha * m2_float
    _, binary = cv2.threshold((interpolated * 255).astype(np.uint8), 127, 255, cv2.THRESH_BINARY)
    return binary


def interpolate_causal(seq: List[Optional[np.ndarray]], image_size: Tuple[int, int]) -> List[np.ndarray]:
    result, last = [], None
    for m in seq:
        if m is not None:
            result.append(m)
            last = m
        elif last is not None:
            result.append(last.copy())
        else:
            result.append(np.zeros(image_size, dtype=np.uint8))
    return result


def interpolate_bidirectional(seq: List[Optional[np.ndarray]], image_size: Tuple[int, int]) -> List[np.ndarray]:
    result = []
    for i, m in enumerate(seq):
        if m is not None:
            result.append(m)
            continue
        prev_idx, next_idx = None, None
        for j in range(i - 1, -1, -1):
            if seq[j] is not None:
                prev_idx = j
                break
        for j in range(i + 1, len(seq)):
            if seq[j] is not None:
                next_idx = j
                break
        if prev_idx is None and next_idx is None:
            result.append(np.zeros(image_size, dtype=np.uint8))
        elif prev_idx is None:
            result.append(seq[next_idx].copy())
        elif next_idx is None:
            result.append(seq[prev_idx].copy())
        else:
            alpha = (i - prev_idx) / (next_idx - prev_idx)
            result.append(pearson_interpolation(seq[prev_idx], seq[next_idx], alpha))
    return result


class MIMN(nn.Module):
    def __init__(self, num_hidden, height, width, filter_size=3):
        super(MIMN, self).__init__()
        self.num_hidden = num_hidden
        self.padding = (filter_size - 1) // 2

        num_groups = max(1, (4 * num_hidden) // 32)

        self.conv_h = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.conv_x = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )

        self.ct_weight = nn.Parameter(torch.randn(2 * num_hidden, height, width) * 0.1)
        self.oc_weight = nn.Parameter(torch.randn(num_hidden, height, width) * 0.1)

    def forward(self, x, h_t, c_t):
        h_concat = self.conv_h(h_t)
        x_concat = self.conv_x(x)

        i_h, g_h, f_h, o_h = torch.split(h_concat, self.num_hidden, dim=1)
        i_x, g_x, f_x, o_x = torch.split(x_concat, self.num_hidden, dim=1)

        ct_activation = c_t.repeat(1, 2, 1, 1) * self.ct_weight
        i_c, f_c = torch.split(ct_activation, self.num_hidden, dim=1)

        i_ = i_x + i_h + i_c
        f_ = f_x + f_h + f_c
        g_ = g_x + g_h
        o_ = o_x + o_h

        i_ = torch.sigmoid(i_)
        f_ = torch.sigmoid(f_ + 1.0)
        g_ = torch.tanh(g_)

        c_new = f_ * c_t + i_ * g_

        o_c = c_new * self.oc_weight
        o_ = torch.sigmoid(o_ + o_c)

        h_new = o_ * torch.tanh(c_new)

        return h_new, c_new


class MIMBlock(nn.Module):
    def __init__(self, num_hidden, height, width, filter_size=3):
        super(MIMBlock, self).__init__()
        self.num_hidden = num_hidden
        self.padding = (filter_size - 1) // 2
        num_groups = max(1, (4 * num_hidden) // 32)
        num_groups_3 = max(1, (3 * num_hidden) // 32)

        self.mims_h = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.mims_x = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.mims_ct_weight = nn.Parameter(torch.randn(2 * num_hidden, height, width) * 0.1)
        self.mims_oc_weight = nn.Parameter(torch.randn(num_hidden, height, width) * 0.1)

        self.t_cc = nn.Sequential(
            nn.Conv2d(num_hidden, 3 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups_3, 3 * num_hidden)
        )
        self.s_cc = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.x_cc = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )

        self.c_reduce = nn.Conv2d(2 * num_hidden, num_hidden, 1, 1, 0)

    def MIMS(self, x, h_t, c_t):
        h_concat = self.mims_h(h_t)
        i_h, g_h, f_h, o_h = torch.split(h_concat, self.num_hidden, dim=1)

        ct_activation = c_t.repeat(1, 2, 1, 1) * self.mims_ct_weight
        i_c, f_c = torch.split(ct_activation, self.num_hidden, dim=1)

        i_ = i_h + i_c
        f_ = f_h + f_c
        g_ = g_h
        o_ = o_h

        if x is not None:
            x_concat = self.mims_x(x)
            i_x, g_x, f_x, o_x = torch.split(x_concat, self.num_hidden, dim=1)
            i_ = i_ + i_x
            f_ = f_ + f_x
            g_ = g_ + g_x
            o_ = o_ + o_x

        i_ = torch.sigmoid(i_)
        f_ = torch.sigmoid(f_ + 1.0)
        g_ = torch.tanh(g_)

        c_new = f_ * c_t + i_ * g_

        o_c = c_new * self.mims_oc_weight
        h_new = torch.sigmoid(o_ + o_c) * torch.tanh(c_new)

        return h_new, c_new

    def forward(self, x, diff_h, h, c, m, convlstm_c):
        t_cc = self.t_cc(h)
        s_cc = self.s_cc(m)
        x_cc = self.x_cc(x)

        i_s, g_s, f_s, o_s = torch.split(s_cc, self.num_hidden, dim=1)
        i_t, g_t, o_t = torch.split(t_cc, self.num_hidden, dim=1)
        i_x, g_x, f_x, o_x = torch.split(x_cc, self.num_hidden, dim=1)

        i = torch.sigmoid(i_x + i_t)
        i_ = torch.sigmoid(i_x + i_s)
        g = torch.tanh(g_x + g_t)
        g_ = torch.tanh(g_x + g_s)
        f_ = torch.sigmoid(f_x + f_s + 1.0)
        o = torch.sigmoid(o_x + o_t + o_s)

        new_m = f_ * m + i_ * g_

        c, new_convlstm_c = self.MIMS(diff_h, c, convlstm_c)
        new_c = c + i * g

        cell = torch.cat([new_c, new_m], dim=1)
        cell = self.c_reduce(cell)
        new_h = o * torch.tanh(cell)

        return new_h, new_c, new_m, new_convlstm_c


class SpatioTemporalLSTMCell(nn.Module):
    def __init__(self, input_channels, num_hidden, filter_size=3):
        super(SpatioTemporalLSTMCell, self).__init__()
        self.num_hidden = num_hidden
        self.padding = (filter_size - 1) // 2
        num_groups = max(1, (4 * num_hidden) // 32)

        self.conv_t = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.conv_s = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.conv_x = nn.Sequential(
            nn.Conv2d(input_channels, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.conv_last = nn.Conv2d(2 * num_hidden, num_hidden, 1, 1, 0)

    def forward(self, x, h, c, m):
        t_cc = self.conv_t(h)
        s_cc = self.conv_s(m)
        x_cc = self.conv_x(x)

        i_s, g_s, f_s, o_s = torch.split(s_cc, self.num_hidden, dim=1)
        i_t, g_t, f_t, o_t = torch.split(t_cc, self.num_hidden, dim=1)
        i_x, g_x, f_x, o_x = torch.split(x_cc, self.num_hidden, dim=1)

        i = torch.sigmoid(i_x + i_t)
        i_ = torch.sigmoid(i_x + i_s)
        g = torch.tanh(g_x + g_t)
        g_ = torch.tanh(g_x + g_s)
        f = torch.sigmoid(f_x + f_t + 1.0)
        f_ = torch.sigmoid(f_x + f_s + 1.0)
        o = torch.sigmoid(o_x + o_t + o_s)

        new_m = f_ * m + i_ * g_
        new_c = f * c + i * g

        cell = torch.cat([new_c, new_m], dim=1)
        cell = self.conv_last(cell)
        new_h = o * torch.tanh(cell)

        return new_h, new_c, new_m


class ReservoirConvLSTM(nn.Module):
    def __init__(self, input_dim=1, num_hidden=16, num_layers=3, shape=(320, 320)):
        super().__init__()
        self.num_layers = num_layers
        self.num_hidden = num_hidden
        self.shape = shape

        self.stlstm_layer = SpatioTemporalLSTMCell(input_dim, num_hidden)

        self.mim_layers = nn.ModuleList()
        self.mimn_layers = nn.ModuleList()

        for i in range(1, num_layers):
            self.mim_layers.append(
                MIMBlock(num_hidden, shape[0], shape[1])
            )
            self.mimn_layers.append(
                MIMN(num_hidden, shape[0], shape[1])
            )

        self.conv_last = nn.Conv2d(num_hidden, input_dim, 1, 1, 0)

    def forward(self, seq):
        device = seq.device
        B, T, C, H, W = seq.shape

        h_t = [torch.zeros(B, self.num_hidden, H, W).to(device) for _ in range(self.num_layers)]
        c_t = [torch.zeros(B, self.num_hidden, H, W).to(device) for _ in range(self.num_layers)]
        convlstm_c_t = [torch.zeros(B, self.num_hidden, H, W).to(device) for _ in range(self.num_layers)]

        diff_h_t = [torch.zeros(B, self.num_hidden, H, W).to(device) for _ in range(self.num_layers - 1)]
        diff_c_t = [torch.zeros(B, self.num_hidden, H, W).to(device) for _ in range(self.num_layers - 1)]

        st_memory = torch.zeros(B, self.num_hidden, H, W).to(device)

        prev_h0 = None
        gen_images = []

        for t in range(T):
            x = seq[:, t, ...]

            if t == 0:
                prev_h0 = torch.zeros_like(h_t[0])

            h_t[0], c_t[0], st_memory = self.stlstm_layer(x, h_t[0], c_t[0], st_memory)

            for i in range(1, self.num_layers):
                if t > 0:
                    if i == 1:
                        diff_input = h_t[0] - prev_h0
                    else:
                        diff_input = diff_h_t[i-2]

                    diff_h_t[i-1], diff_c_t[i-1] = self.mimn_layers[i-1](diff_input, diff_h_t[i-1], diff_c_t[i-1])
                else:
                    diff_input = torch.zeros(B, self.num_hidden, H, W).to(device)
                    diff_h_t[i-1], diff_c_t[i-1] = self.mimn_layers[i-1](diff_input, diff_h_t[i-1], diff_c_t[i-1])

                h_t[i], c_t[i], st_memory, convlstm_c_t[i] = self.mim_layers[i-1](
                    h_t[i-1], diff_h_t[i-1], h_t[i], c_t[i], st_memory, convlstm_c_t[i]
                )

            prev_h0 = h_t[0].clone()

            x_gen = self.conv_last(h_t[-1])
            gen_images.append(x_gen)

        return torch.sigmoid(gen_images[-1])


def train_yolo_segmentation(config: PipelineConfig) -> YOLO:
    yolo_data_dir = config.processed_dir / "yolo_data"
    for split in ["train", "val"]:
        (yolo_data_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (yolo_data_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    train_imgs = load_split_file(config.split_dir / "train.txt")
    val_imgs = load_split_file(config.split_dir / "val.txt")

    for img_path in train_imgs:
        if img_path.exists():
            shutil.copy(str(img_path), str(yolo_data_dir / "images" / "train" / img_path.name))
            label_path = config.yolo_label_dir / (img_path.stem + ".txt")
            if label_path.exists():
                shutil.copy(str(label_path), str(yolo_data_dir / "labels" / "train" / label_path.name))

    for img_path in val_imgs:
        if img_path.exists():
            shutil.copy(str(img_path), str(yolo_data_dir / "images" / "val" / img_path.name))
            label_path = config.yolo_label_dir / (img_path.stem + ".txt")
            if label_path.exists():
                shutil.copy(str(label_path), str(yolo_data_dir / "labels" / "val" / label_path.name))

    data_yaml = {"path": str(yolo_data_dir.absolute()), "train": "images/train", "val": "images/val", "names": {0: "water"}}
    yaml_path = yolo_data_dir / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f)

    print(f"YOLO dataset: {len(train_imgs)} train, {len(val_imgs)} val images")
    model = YOLO("yolo11x-seg.pt")
    model.train(data=str(yaml_path), epochs=config.yolo_epochs, imgsz=config.yolo_image_size, batch=8, patience=20,
                save=True, project=str(config.processed_dir / "yolo_runs"), name="water_seg", exist_ok=True, verbose=True)
    best_path = config.processed_dir / "yolo_runs" / "water_seg" / "weights" / "best.pt"
    if best_path.exists():
        model = YOLO(str(best_path))
    return model


def yolo_predict_mask(model: YOLO, image_path: Path, config: PipelineConfig) -> Optional[np.ndarray]:
    results = model.predict(str(image_path), imgsz=config.yolo_image_size, verbose=False)
    if results and results[0].masks is not None:
        mask = results[0].masks.data[0].cpu().numpy()
        mask = cv2.resize(mask, config.image_size, interpolation=cv2.INTER_NEAREST)
        return (mask > 0.5).astype(np.uint8) * 255
    return None


def evaluate_yolo_on_test(model: YOLO, config: PipelineConfig) -> Dict[str, float]:
    test_imgs = load_split_file(config.split_dir / "test.txt")
    ious, f1s = [], []
    for img_path in test_imgs:
        if not img_path.exists():
            continue
        gt_mask_path = config.convlstm_mask_dir / img_path.name
        if not gt_mask_path.exists():
            continue
        gt_mask = cv2.imread(str(gt_mask_path), cv2.IMREAD_GRAYSCALE)
        pred_mask = yolo_predict_mask(model, img_path, config)
        if pred_mask is not None:
            ious.append(compute_iou(gt_mask, pred_mask))
            f1s.append(compute_f1(gt_mask, pred_mask))
    results = {"mean_iou": np.mean(ious) if ious else 0.0, "mean_f1": np.mean(f1s) if f1s else 0.0, "num_samples": len(ious)}
    print(f"YOLO Test: IoU={results['mean_iou']:.4f}, F1={results['mean_f1']:.4f} ({results['num_samples']} samples)")
    return results


def compute_iou(mask1: np.ndarray, mask2: np.ndarray) -> float:
    m1 = (mask1 > 127).astype(bool) if mask1.max() > 1 else mask1.astype(bool)
    m2 = (mask2 > 127).astype(bool) if mask2.max() > 1 else mask2.astype(bool)
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    return inter / (union + 1e-8)


def compute_f1(mask1: np.ndarray, mask2: np.ndarray) -> float:
    m1 = (mask1 > 127).astype(bool) if mask1.max() > 1 else mask1.astype(bool)
    m2 = (mask2 > 127).astype(bool) if mask2.max() > 1 else mask2.astype(bool)
    tp = np.logical_and(m1, m2).sum()
    fp = np.logical_and(m1, ~m2).sum()
    fn = np.logical_and(~m1, m2).sum()
    prec = tp / (tp + fp + 1e-8)
    rec = tp / (tp + fn + 1e-8)
    return 2 * prec * rec / (prec + rec + 1e-8)


print("Models defined: ReservoirConvLSTM, YOLOv11, MSE loss")

In [ ]:
class TemporalMaskDataset(Dataset):
    def __init__(self, files: List[Path], seq_len: int = 5):
        self.files = list(files)
        self.seq_len = seq_len

    def __len__(self): return max(1, len(self.files) - self.seq_len)

    def __getitem__(self, idx):
        idx = min(idx, max(0, len(self.files) - self.seq_len - 1))
        masks = [cv2.imread(str(self.files[idx + k]), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
                 for k in range(self.seq_len + 1)]
        return (torch.from_numpy(np.stack([m[None] for m in masks[:-1]])).float(),
                torch.from_numpy(masks[-1][None]).float())


def train_convlstm(config: PipelineConfig) -> nn.Module:
    torch.cuda.empty_cache()
    gc.collect()
    train_files = load_split_file(config.split_dir / "train_masks.txt")
    val_files = load_split_file(config.split_dir / "val_masks.txt")
    if not train_files:
        all_files = sorted(config.dense_mask_dir.glob("*.png"))
        train_files, val_files = train_test_split(all_files, test_size=0.2, shuffle=False) if all_files else ([], [])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ReservoirConvLSTM().to(device)
    loader = DataLoader(TemporalMaskDataset(train_files, config.convlstm_sequence_length), config.convlstm_batch_size, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=config.convlstm_learning_rate, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, config.convlstm_epochs)
    mse = nn.MSELoss()

    best_loss, best_state = float("inf"), None
    print(f"Training ConvLSTM: {len(train_files)} train, {len(val_files)} val masks")

    for epoch in range(config.convlstm_epochs):
        model.train()
        losses = []
        opt.zero_grad()
        for i, (seq, target) in enumerate(loader):
            pred = model(seq.to(device))
            loss = mse(pred, target.to(device)) / config.convlstm_accumulation_steps
            loss.backward()
            if (i + 1) % config.convlstm_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                opt.zero_grad()
            losses.append(loss.item() * config.convlstm_accumulation_steps)
        sched.step()

        model.eval()
        with torch.no_grad():
            val_loader = DataLoader(TemporalMaskDataset(val_files, config.convlstm_sequence_length), config.convlstm_batch_size)
            val_loss = np.mean([mse(model(x.to(device)), y.to(device)).item() for x, y in val_loader])
        if val_loss < best_loss:
            best_loss, best_state = val_loss, {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}: train={np.mean(losses):.4f}, val={val_loss:.4f}")

    if best_state:
        model.load_state_dict(best_state)
    print(f"Best val loss: {best_loss:.4f}")
    return model


REAL_TIMESTAMPS: set = set()

def build_dense_masks(records: List[Dict[str, Path]], config: PipelineConfig) -> Tuple[List[Path], set]:
    global REAL_TIMESTAMPS
    sorted_recs = sort_records_by_timestamp(records)
    dated = [(parse_timestamp(Path(r["mask"])), r["mask"]) for r in sorted_recs]
    dated = [(ts, p) for ts, p in dated if ts is not None]
    REAL_TIMESTAMPS = {ts for ts, _ in dated}
    if not dated:
        return sorted(config.convlstm_mask_dir.glob("*.png")), set()

    paths_list = [Path(p) for _, p in dated]
    train_paths, _ = train_test_split(paths_list, train_size=config.train_ratio, shuffle=False)
    ts_to_mask = {ts: Path(p) for ts, p in dated}
    train_ts_set = {ts for ts, p in dated if Path(p) in train_paths}

    monthly = pd.date_range(start=min(ts for ts, _ in dated), end=max(ts for ts, _ in dated), freq="MS")
    seq = [cv2.resize(cv2.imread(str(ts_to_mask[ts]), cv2.IMREAD_GRAYSCALE), config.image_size, interpolation=cv2.INTER_NEAREST)
           if ts in ts_to_mask else None for ts in monthly]

    train_end_ts = max(train_ts_set) if train_ts_set else monthly[0]
    train_end = sum(1 for ts in monthly if ts <= train_end_ts)

    print(f"Interpolation: {train_end} train months (bidirectional), {len(monthly) - train_end} val/test months (causal)")

    filled = interpolate_bidirectional(seq[:train_end], config.image_size) + interpolate_causal(seq[train_end:], config.image_size)

    out_paths, interpolated = [], set()
    for ts, m in zip(monthly, filled):
        out = config.dense_mask_dir / f"mask_{ts.strftime('%Y-%m')}.png"
        cv2.imwrite(str(out), m)
        out_paths.append(out)
        if ts not in REAL_TIMESTAMPS:
            interpolated.add(out)
    print(f"Generated {len(out_paths)} dense masks ({len(interpolated)} interpolated)")
    return out_paths, interpolated

DENSE_MASKS, INTERPOLATED_MASKS = build_dense_masks(records, cfg)
splits = create_dataset_splits(records, cfg, DENSE_MASKS, INTERPOLATED_MASKS)

In [ ]:
class InterpolationMethod:
    def __init__(self, name: str):
        self.name = name

    def interpolate(self, m1: np.ndarray, m2: np.ndarray, alpha: float) -> np.ndarray:
        raise NotImplementedError


class PearsonInterp(InterpolationMethod):
    def __init__(self):
        super().__init__("pearson")

    def interpolate(self, m1: np.ndarray, m2: np.ndarray, alpha: float) -> np.ndarray:
        return pearson_interpolation(m1, m2, alpha)

INTERPOLATION_METHODS = {
    'pearson': PearsonInterp(),
}


def generate_interpolated_masks_safe(train_mask_paths: List[Path], method: InterpolationMethod,
                                     config: PipelineConfig, interp_per_gap: int = 3) -> List[Path]:
    if len(train_mask_paths) < 2:
        return list(train_mask_paths)

    sorted_paths = sorted(train_mask_paths, key=lambda p: (parse_timestamp(p) or pd.Timestamp.max, p.name))

    output_dir = config.processed_dir / f"interp_{method.name}"
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(exist_ok=True)

    all_paths = []
    for i, p in enumerate(sorted_paths):
        out = output_dir / f"orig_{i:03d}_{p.name}"
        shutil.copy(str(p), str(out))
        all_paths.append(out)

    interp_paths = []
    for i in range(len(sorted_paths) - 1):
        m1 = cv2.imread(str(sorted_paths[i]), cv2.IMREAD_GRAYSCALE)
        m2 = cv2.imread(str(sorted_paths[i + 1]), cv2.IMREAD_GRAYSCALE)

        if m1 is None or m2 is None:
            continue

        m1 = cv2.resize(m1, config.image_size, interpolation=cv2.INTER_NEAREST)
        m2 = cv2.resize(m2, config.image_size, interpolation=cv2.INTER_NEAREST)

        for j in range(1, interp_per_gap + 1):
            alpha = j / (interp_per_gap + 1)
            try:
                interp = method.interpolate(m1, m2, alpha)
                out_path = output_dir / f"interp_{i:03d}_{j:02d}.png"
                cv2.imwrite(str(out_path), interp)
                interp_paths.append((i, j, out_path))
            except Exception as e:
                print(f"    Warning: interpolation failed at {i},{j}: {e}")

    for i, j, path in interp_paths:
        insert_idx = i + 1 + (j - 1)
        all_paths.insert(min(insert_idx, len(all_paths)), path)

    all_paths = sorted(all_paths, key=lambda p: p.name)
    print(f"    Generated {len(interp_paths)} interpolated + {len(sorted_paths)} original = {len(all_paths)} total")
    return all_paths

def rmse(gt: np.ndarray, pred: np.ndarray) -> float:
    return np.sqrt(np.mean((gt - pred) ** 2))

def mae(gt: np.ndarray, pred: np.ndarray) -> float:
    return np.mean(np.abs(gt - pred))

def mask_to_water_level(mask: np.ndarray, pixel_area_km2: float = 0.0001) -> float:
    water_pixels = (mask > 127).sum() if mask.max() > 1 else (mask > 0.5).sum()
    return water_pixels * pixel_area_km2

In [ ]:
import os
import pandas as pd
import yaml
import shutil
from ultralytics import YOLO
from pathlib import Path

def train_evaluate_single_yolo(config: PipelineConfig, model_name: str) -> Tuple[pd.DataFrame, YOLO]:
    print("=" * 60)
    print(f"TRAINING SINGLE YOLO MODEL: {model_name}")
    print("=" * 60)

    yolo_data_dir = config.processed_dir / "yolo_benchmark_data"
    for split in ["train", "val", "test"]:
        (yolo_data_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (yolo_data_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    def copy_split(split_name, text_file_name):
        img_paths = load_split_file(config.split_dir / text_file_name)
        count = 0
        for img_path in img_paths:
            if img_path.exists():
                shutil.copy(str(img_path), str(yolo_data_dir / "images" / split_name / img_path.name))
                label_path = config.yolo_label_dir / (img_path.stem + ".txt")
                if label_path.exists():
                    shutil.copy(str(label_path), str(yolo_data_dir / "labels" / split_name / label_path.name))
                count += 1
        return count

    n_train = copy_split("train", "train.txt")
    n_val = copy_split("val", "val.txt")
    n_test = copy_split("test", "test.txt")

    print(f"Dataset Prepared: {n_train} Train, {n_val} Val, {n_test} Test images")

    data_yaml = {
        "path": str(yolo_data_dir.absolute()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "water"}
    }
    yaml_path = yolo_data_dir / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f)

    results_data = []

    print(f"\n--- Processing {model_name} ---")
    run_name = model_name.replace(".pt", "")

    best_weight_path = config.processed_dir / "yolo_benchmark" / run_name / "weights" / "best.pt"

    if best_weight_path.exists():
        print(f"Found existing weights at {best_weight_path}. Skipping training...")
        try:
            model = YOLO(str(best_weight_path))
        except Exception as e:
            print(f"Could not load existing weights {best_weight_path}: {e}")
            model = YOLO(model_name)
    else:
        print(f"No existing weights found. Training {model_name} from scratch...")
        try:
            model = YOLO(model_name)
        except Exception as e:
            print(f"Could not load {model_name}: {e}")
            raise e

        model.train(data=str(yaml_path),
                    epochs=config.yolo_epochs,
                    imgsz=config.yolo_image_size,
                    batch=8,
                    patience=50,
                    project=str(config.processed_dir / "yolo_benchmark"),
                    name=run_name,
                    exist_ok=True,
                    verbose=False)

        if best_weight_path.exists():
            model = YOLO(str(best_weight_path))

    print(f"Evaluating {model_name} on Test Set...")
    metrics = model.val(split='test', verbose=False)

    precision = metrics.seg.mp
    recall = metrics.seg.mr
    map50 = metrics.seg.map50
    map5095 = metrics.seg.map

    try:
        info = model.info(verbose=False)
        if info and isinstance(info, (tuple, list)):
            params_m = info[1] / 1e6
            gflops = info[3]
        else:
            params = sum(p.numel() for p in model.model.parameters())
            params_m = params / 1e6
            gflops = 0.0
    except Exception:
        params = sum(p.numel() for p in model.model.parameters())
        params_m = params / 1e6
        gflops = 0.0

    size_mb = os.path.getsize(best_weight_path) / (1024 * 1024) if best_weight_path.exists() else 0

    inference_time = metrics.speed['inference']
    fps = 1000.0 / inference_time if inference_time > 0 else 0

    results_data.append({
        "Model": model_name,
        "P": round(precision, 3),
        "R": round(recall, 3),
        "mAP@0.5": round(map50, 3),
        "mAP@0.5:0.95": round(map5095, 3),
        "Params (M)": round(params_m, 2),
        "Size (MB)": round(size_mb, 1),
        "GFLOPs": round(gflops, 1),
        "FPS": round(fps, 1)
    })

    df = pd.DataFrame(results_data)
    return df, model

target_model = "yolo11x-seg.pt"

result_df, trained_yolo_model = train_evaluate_single_yolo(cfg, target_model)

print("\n" + "="*80)
print("YOLO MODEL PERFORMANCE")
print("="*80)
print(result_df.to_markdown(index=False, numalign="center", stralign="center"))

all_yolo_models = {target_model: trained_yolo_model}
benchmark_df = result_df

In [ ]:
def get_yolo_predicted_sequence(model: YOLO, mask_paths: List[Path], config: PipelineConfig) -> List[np.ndarray]:
    yolo_masks = []

    available_images = list(config.yolo_image_dir.glob("*.png"))
    timestamp_to_image = {}

    for img_path in available_images:
        ts = parse_timestamp(img_path)
        if ts is not None:
            timestamp_to_image[ts] = img_path

    print(f"Generating YOLO masks for {len(mask_paths)} images...")

    for mask_path in tqdm(mask_paths, desc="YOLO Inference"):
        img_path = None

        direct_path = config.yolo_image_dir / mask_path.name
        if direct_path.exists():
            img_path = direct_path
        else:
            ts = parse_timestamp(mask_path)
            if ts is not None and ts in timestamp_to_image:
                img_path = timestamp_to_image[ts]
            else:
                if ts is not None:
                    closest_ts = min(timestamp_to_image.keys(),
                                     key=lambda x: abs((x - ts).days),
                                     default=None)
                    if closest_ts is not None:
                        img_path = timestamp_to_image[closest_ts]

        if img_path is not None and img_path.exists():
            pred_mask = yolo_predict_mask(model, img_path, config)
        else:
            pred_mask = None

        if pred_mask is None:
            pred_mask = np.zeros(config.image_size, dtype=np.uint8)

        yolo_masks.append(pred_mask.astype(np.float32) / 255.0)

    return yolo_masks

In [ ]:
from sklearn.model_selection import KFold
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm
import gc

def evaluate_fold_scenarios_pipeline(conv_model, yolo_model, val_files, config):
    conv_model.eval()
    device = next(conv_model.parameters()).device

    gt_masks = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).astype(np.float32)/255.0 for p in val_files]

    print(f"  Generating YOLO masks for {len(val_files)} images...")
    input_masks = get_yolo_predicted_sequence(yolo_model, val_files, config)

    results = {}

    for start_len in [3, 4, 5]:
        errors_mae = []
        errors_mse = []
        min_req = start_len + 1

        loop_range = range(len(input_masks) - min_req + 1)
        if len(loop_range) > 0:
            for i in tqdm(loop_range, desc=f"  Eval Scen (Input {start_len})", leave=False):
                with torch.no_grad():
                    seq_np = np.stack([m[None] for m in input_masks[i : i + start_len]])
                    input_tensor = torch.from_numpy(seq_np).float().unsqueeze(0).to(device)

                    with autocast():
                        pred = conv_model(input_tensor).squeeze().cpu().float().numpy()

                    gt = gt_masks[i + start_len]

                    pred_lvl = mask_to_level(pred, config)
                    gt_lvl = mask_to_level(gt, config)

                    err = abs(pred_lvl - gt_lvl)
                    errors_mae.append(err)
                    errors_mse.append(err ** 2)

        if errors_mae:
            results[f"{start_len} images"] = {
                "MAE": np.mean(errors_mae),
                "RMSE": np.sqrt(np.mean(errors_mse))
            }
        else:
            results[f"{start_len} images"] = {"MAE": 0.0, "RMSE": 0.0}

    return results

def mask_to_level(mask: np.ndarray, config: PipelineConfig) -> float:
    water_pixels = (mask > 0.5).sum() if mask.max() <= 1 else (mask > 127).sum()
    return water_pixels * config.pixel_area_km2

k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=False)

all_mask_paths = load_split_file(cfg.split_dir / "train_masks.txt") + \
                 load_split_file(cfg.split_dir / "val_masks.txt") + \
                 load_split_file(cfg.split_dir / "test_masks.txt")
all_mask_paths = sorted(list(set(all_mask_paths)), key=lambda p: p.name)

model_comparison_results = {}
trained_convlstm_models = {}

for yolo_name, yolo_model in all_yolo_models.items():
    print(f"\n{'='*80}")
    print(f"EVALUATING YOLO MODEL: {yolo_name}")
    print(f"{'='*80}")

    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(all_mask_paths)):
        torch.cuda.empty_cache()
        gc.collect()

        print(f"\n--- Fold {fold + 1}/{k_folds} ---")

        fold_train_paths = [all_mask_paths[i] for i in train_idx]
        fold_val_paths = [all_mask_paths[i] for i in val_idx]

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = ReservoirConvLSTM(num_hidden=32, num_layers=3, shape=cfg.image_size).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.convlstm_learning_rate)
        scaler = GradScaler()
        criterion = nn.MSELoss()

        dataset = TemporalMaskDataset(fold_train_paths, cfg.convlstm_sequence_length)
        loader = DataLoader(dataset, batch_size=cfg.convlstm_batch_size, shuffle=True, drop_last=True)

        model.train()
        k_epochs = 15

        print(f"  Training ConvLSTM ({len(fold_train_paths)} samples)...")
        for epoch in range(k_epochs):
            epoch_loss = 0
            pbar = tqdm(loader, desc=f"  Epoch {epoch+1}/{k_epochs}", leave=True)

            for seq, target in pbar:
                seq, target = seq.to(device), target.to(device)

                optimizer.zero_grad()

                with autocast():
                    pred = model(seq)
                    loss = criterion(pred, target)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

                epoch_loss += loss.item()
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})

            avg_loss = epoch_loss / len(loader)

        print(f"  Evaluating Fold {fold+1}...")
        metrics = evaluate_fold_scenarios_pipeline(model, yolo_model, fold_val_paths, cfg)

        fold_record = {"Fold": f"Fold {fold + 1}"}
        for input_len, vals in metrics.items():
            fold_record[f"MAE_{input_len}"] = vals["MAE"]
            fold_record[f"RMSE_{input_len}"] = vals["RMSE"]

        fold_results.append(fold_record)
        print(f"  > Fold {fold+1} Results: {metrics}")

    trained_convlstm_models[yolo_name] = model

    df_kfold = pd.DataFrame(fold_results)
    avg_row = {"Fold": "Average"}
    for col in df_kfold.columns:
        if col != "Fold":
            avg_row[col] = df_kfold[col].mean()
    fold_results.append(avg_row)

    model_comparison_results[yolo_name] = pd.DataFrame(fold_results)

print("\n" + "="*100)
print("K-FOLD RESULTS SUMMARY")
print("="*100)

for yolo_name, df_result in model_comparison_results.items():
    print(f"\n{'='*80}")
    print(f"Results for: {yolo_name}")
    print("="*80)

    ordered_cols = ["Fold", "MAE_3 images", "MAE_4 images", "MAE_5 images",
                    "RMSE_3 images", "RMSE_4 images", "RMSE_5 images"]
    df_display = df_result[ordered_cols].copy()
    df_display.columns = ["Input", "MAE (3)", "MAE (4)", "MAE (5)",
                          "RMSE (3)", "RMSE (4)", "RMSE (5)"]
    print(df_display.to_markdown(index=False, floatfmt=".4f", numalign="center"))

print("\n" + "="*100)
print("AVERAGE PERFORMANCE SUMMARY")
print("="*100)

summary_data = []
for yolo_name, df_result in model_comparison_results.items():
    avg_row = df_result[df_result["Fold"] == "Average"].iloc[0]
    summary_data.append({
        "YOLO Model": yolo_name,
        "MAE (3 img)": avg_row.get("MAE_3 images", 0),
        "MAE (4 img)": avg_row.get("MAE_4 images", 0),
        "MAE (5 img)": avg_row.get("MAE_5 images", 0),
        "RMSE (3 img)": avg_row.get("RMSE_3 images", 0),
        "RMSE (4 img)": avg_row.get("RMSE_4 images", 0),
        "RMSE (5 img)": avg_row.get("RMSE_5 images", 0),
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_markdown(index=False, floatfmt=".4f", numalign="center"))

In [ ]:

def evaluate_recursive_steps_pipeline(conv_model: nn.Module, yolo_model: YOLO, config: PipelineConfig,
                                      start_lengths: List[int] = [3, 4, 5],
                                      forecast_horizon: int = 3) -> pd.DataFrame:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    conv_model.to(device)
    conv_model.eval()

    test_masks_paths = load_split_file(config.split_dir / "test_masks.txt")
    test_masks_paths = sorted(test_masks_paths, key=lambda p: (parse_timestamp(p) or pd.Timestamp.max, p.name))

    gt_masks = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).astype(np.float32)/255.0 for p in test_masks_paths]

    print("Generating YOLO inputs for test set...")
    yolo_inputs = get_yolo_predicted_sequence(yolo_model, test_masks_paths, config)

    detailed_results = []

    with torch.no_grad():
        for start_len in start_lengths:
            print(f"Processing Pipeline Input Length: {start_len} images...")
            step_metrics = {s: {'rmses': [], 'maes': [], 'ious': []} for s in range(1, forecast_horizon + 1)}
            min_required = start_len + forecast_horizon

            for i in range(len(yolo_inputs) - min_required + 1):

                current_seq = [m.copy() for m in yolo_inputs[i : i + start_len]]

                for step in range(1, forecast_horizon + 1):
                    input_np = np.stack([m[None] for m in current_seq[-start_len:]])
                    input_tensor = torch.from_numpy(input_np).float().unsqueeze(0).to(device)

                    pred_tensor = conv_model(input_tensor)
                    pred_np = pred_tensor.squeeze().cpu().numpy()

                    gt_idx = i + start_len + step - 1
                    gt_np = gt_masks[gt_idx]

                    pred_255 = pred_np * 255
                    gt_255 = gt_np * 255
                    pred_bin = (pred_255 > 127).astype(np.uint8)
                    gt_bin = (gt_255 > 127).astype(np.uint8)

                    rmse = np.sqrt(np.mean((gt_255 - pred_255) ** 2))
                    mae_val = np.mean(np.abs(gt_255 - pred_255))
                    iou = compute_iou(gt_bin, pred_bin)

                    step_metrics[step]['rmses'].append(rmse)
                    step_metrics[step]['maes'].append(mae_val)
                    step_metrics[step]['ious'].append(iou)

                    current_seq.append(pred_np)

            for step in range(1, forecast_horizon + 1):
                m = step_metrics[step]
                if m['rmses']:
                    detailed_results.append({
                        "Input Images": f"{start_len} images",
                        "Forecast Step": f"Step {step}",
                        "MAE": np.mean(m['maes']),
                        "RMSE": np.mean(m['rmses']),
                        "IoU": np.mean(m['ious']),
                        "Count": len(m['rmses'])
                    })

    df = pd.DataFrame(detailed_results)
    return df

print("\n" + "="*80)
print("RECURSIVE FORECASTING EVALUATION (SINGLE MODEL PIPELINE)")
print("="*80)
yolo_name = list(all_yolo_models.keys())[0]
yolo_model = all_yolo_models[yolo_name]
if yolo_name in trained_convlstm_models:
    convlstm_model = trained_convlstm_models[yolo_name]
else:
    print(f"Training ConvLSTM for {yolo_name}...")
    convlstm_model = train_convlstm(cfg)

print(f"\nEvaluating Recursive Forecasting for: {yolo_name}")
step_results = evaluate_recursive_steps_pipeline(convlstm_model, yolo_model, cfg)

print(f"\nResults for {yolo_name}:")
print("-" * 85)
print(f"{'Input Length':<15} {'Forecast Step':<15} {'MAE':<10} {'RMSE':<10} {'IoU':<10}")
print("-" * 85)
for _, row in step_results.iterrows():
    print(f"{row['Input Images']:<15} {row['Forecast Step']:<15} {row['MAE']:.4f}     "
          f"{row['RMSE']:.4f}     {row['IoU']:.4f}")

print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
print(step_results.to_markdown(index=False, floatfmt=".4f", numalign="center"))

In [ ]:
def get_water_level(mask: np.ndarray, config: PipelineConfig) -> float:
    water_pixels = (mask > 0.5).sum() if mask.max() <= 1 else (mask > 127).sum()
    return water_pixels * config.pixel_area_km2

def evaluate_water_levels_pipeline(conv_model, yolo_model, config,
                                   start_lengths=[3, 4, 5], forecast_horizon=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    conv_model.to(device)
    conv_model.eval()

    test_paths = load_split_file(config.split_dir / "test_masks.txt")
    test_paths = sorted(test_paths, key=lambda p: (parse_timestamp(p) or pd.Timestamp.max, p.name))

    gt_masks = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).astype(np.float32)/255.0 for p in test_paths]

    print("Generating YOLO inputs for water level evaluation...")
    yolo_inputs = get_yolo_predicted_sequence(yolo_model, test_paths, config)

    results = []

    with torch.no_grad():
        for start_len in start_lengths:
            print(f"Water Level Eval (Input: {start_len})...")
            step_metrics = {s: {'errors': []} for s in range(1, forecast_horizon + 1)}
            min_req = start_len + forecast_horizon

            for i in range(len(yolo_inputs) - min_req + 1):
                current_seq = [m.copy() for m in yolo_inputs[i : i + start_len]]

                for step in range(1, forecast_horizon + 1):
                    input_np = np.stack([m[None] for m in current_seq[-start_len:]])
                    input_tensor = torch.from_numpy(input_np).float().unsqueeze(0).to(device)

                    pred = conv_model(input_tensor).squeeze().cpu().numpy()

                    gt_idx = i + start_len + step - 1
                    level_gt = get_water_level(gt_masks[gt_idx], config)
                    level_pred = get_water_level(pred, config)

                    step_metrics[step]['errors'].append(abs(level_gt - level_pred))

                    current_seq.append(pred)

            for step in range(1, forecast_horizon + 1):
                errs = np.array(step_metrics[step]['errors'])
                if len(errs) > 0:
                    results.append({
                        "Input Images": f"{start_len} images",
                        "Forecast Step": f"Step {step}",
                        "MAE (m)": np.mean(errs),
                        "RMSE (m)": np.sqrt(np.mean(errs ** 2))
                    })

    df = pd.DataFrame(results)
    return df
print("\n" + "="*100)
print("WATER LEVEL FORECAST EVALUATION (SINGLE MODEL)")
print("="*100)
yolo_name = list(all_yolo_models.keys())[0]
yolo_model = all_yolo_models[yolo_name]

print(f"\nEvaluating Water Levels with: {yolo_name}")

if yolo_name in trained_convlstm_models:
    convlstm_model = trained_convlstm_models[yolo_name]
else:
    print(f"Training ConvLSTM for {yolo_name}...")
    convlstm_model = train_convlstm(cfg)

wl_results = evaluate_water_levels_pipeline(convlstm_model, yolo_model, cfg)

print(f"\nWater Level Results for {yolo_name}:")
print(wl_results.to_markdown(index=False, floatfmt=".4f", numalign="center"))

In [ ]:
print("="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)

print("\n1. YOLO BENCHMARK (Segmentation Performance):")
print(benchmark_df[["Model", "mAP@0.5", "mAP@0.5:0.95", "FPS"]].to_string(index=False))

print("\n2. K-FOLD AVERAGE (Water Level Prediction Error):")
print(summary_df.to_string(index=False))

print("\n3. BEST PERFORMING MODEL BY METRIC:")
best_map = benchmark_df.loc[benchmark_df["mAP@0.5:0.95"].idxmax(), "Model"]
best_mae_3 = summary_df.loc[summary_df["MAE (3 img)"].idxmin(), "YOLO Model"]
best_mae_5 = summary_df.loc[summary_df["MAE (5 img)"].idxmin(), "YOLO Model"]

print(f"  - Best mAP@0.5:0.95: {best_map}")
print(f"  - Best MAE (3 images): {best_mae_3}")
print(f"  - Best MAE (5 images): {best_mae_5}")

In [ ]:
import torch
from ultralytics import YOLO

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 40)
print("PIPELINE PARAMETER COUNTS")
print("=" * 40)

try:
    convlstm = ReservoirConvLSTM()
    convlstm_params = count_parameters(convlstm)
    print(f"ReservoirConvLSTM (Forecasting): {convlstm_params:,} parameters")
except NameError:
    print("ReservoirConvLSTM class not defined. Please run the model definition cell first.")
except Exception as e:
    print(f"Error counting ConvLSTM parameters: {e}")